# Numerical Example: MaskablePPO Policy Update

## 1. Example parameters

This standalone example illustrates one policy-gradient update for **Taxi-v4**, the environment used in this project. Notebooks 01 and 02 record state **386** with action mask **[1, 1, 0, 1, 0, 0]**. We reuse this exact state, mask, and six-action numbering.

State **386** means taxi position **(3, 4)**, passenger at **Green**, and destination **Yellow** (zero-based coordinates). The state and mask come from the project. **The initial logits, advantage, and illustrative learning rate below are chosen for easy arithmetic; they are not extracted from the saved model or training rollouts.** The old masked probabilities and assumed advantage match the numerical example in notebook 03.

**The main example is the step-by-step numerical calculation in Markdown, which can be followed by hand.** The Python cells verify the arithmetic and execute the same policy update. Only **PyTorch** is required; all example values are explicitly defined.

| Parameter | Value |
|---|---|
| Selected action | 0: South |
| Initial logits | $\log([4,\;3.5,\;5,\;2.5,\;3,\;2])$ |
| Fixed advantage estimate | $A=2.0$ |
| PPO clipping parameter | $\epsilon=0.2$, giving $[0.8,1.2]$ |
| Optimizer and learning rate | SGD, $\alpha=0.1$ |

The logits represent the policy at this one state. Initially, the current policy equals the old policy, as at the start of PPO optimization. The old policy is a detached snapshot and remains fixed. The assumed advantage is held fixed without normalization for this one-sample example. SGD with learning rate 0.1 makes the arithmetic visible. The project's full model uses MlpPolicy with Adam and learning rate 0.0003 (notebook 04), together with batch advantage normalization and gradient norm clipping.

In [1]:
import torch

ACTION_NAMES = ["South", "North", "East", "West", "Pickup", "Dropoff"]
state = 386
action_mask = torch.tensor([1, 1, 0, 1, 0, 0], dtype=torch.bool)
selected_action = 0
advantage = torch.tensor(2.0, dtype=torch.float64)
epsilon = 0.2
learning_rate = 0.1

initial_logits = torch.log(
    torch.tensor([4.0, 3.5, 5.0, 2.5, 3.0, 2.0], dtype=torch.float64)
)
old_logits = initial_logits.detach().clone()
policy_logits = torch.nn.Parameter(initial_logits.clone())

assert action_mask.any() and action_mask[selected_action]
print(f"State: {state}; selected action: {selected_action} ({ACTION_NAMES[selected_action]})")
print(f"Advantage: {advantage.item():.1f}; epsilon: {epsilon}; learning rate: {learning_rate}")

State: 386; selected action: 0 (South)
Advantage: 2.0; epsilon: 0.2; learning rate: 0.1


## 2. Action masking: numerical calculation

For state 386, **South, North, and West are valid**. East, Pickup, and Dropoff are invalid according to the environment's mask.

We replace invalid **logits** with $-\infty$ **before softmax**:

$$
\pi(a\mid s)=
\begin{cases}
\displaystyle\frac{\exp(z_a)}{\sum_{j:\,m_j=1}\exp(z_j)}, & m_a=1,\\
0, & m_a=0.
\end{cases}
$$

Before masking, the weights sum to $4+3.5+5+2.5+3+2=20$, giving

$$
\pi_{\mathrm{unmasked}}
=[0.20,\;0.175,\;0.25,\;0.125,\;0.15,\;0.10].
$$

After masking, only the weights $4$, $3.5$, and $2.5$ remain. Their sum is $10$, so

$$
\begin{aligned}
\pi_{\mathrm{old}}(\mathrm{South}\mid s)&=4/10=0.40,\\
\pi_{\mathrm{old}}(\mathrm{North}\mid s)&=3.5/10=0.35,\\
\pi_{\mathrm{old}}(\mathrm{West}\mid s)&=2.5/10=0.25.
\end{aligned}
$$

Therefore,

$$
\pi_{\mathrm{old}}=[0.40,\;0.35,\;0,\;0.25,\;0,\;0],
\qquad 0.40+0.35+0.25=1.
$$

Invalid actions receive exactly zero probability, so the policy cannot select them. The same mask is reused for the old and current policies. Here $-\infty$ expresses exact exclusion; sb3-contrib uses a very large negative logit.

The following cell verifies these probabilities.

In [2]:
def masked_probabilities(logits):
    return torch.softmax(logits.masked_fill(~action_mask, -torch.inf), dim=0)


with torch.no_grad():
    unmasked_probabilities = torch.softmax(old_logits, dim=0)
    old_probabilities = masked_probabilities(old_logits)
    old_probability = old_probabilities[selected_action].clone()

print(f"{'ID':<3} {'Action':<8} {'Mask':>5} {'Unmasked':>10} {'Masked':>10}")
for i, name in enumerate(ACTION_NAMES):
    print(
        f"{i:<3} {name:<8} {int(action_mask[i]):>5}"
        f" {unmasked_probabilities[i].item():>10.6f}"
        f" {old_probabilities[i].item():>10.6f}"
    )

ID  Action    Mask   Unmasked     Masked
0   South        1   0.200000   0.400000
1   North        1   0.175000   0.350000
2   East         0   0.250000   0.000000
3   West         1   0.125000   0.250000
4   Pickup       0   0.150000   0.000000
5   Dropoff      0   0.100000   0.000000


## 3. PPO clipped objective: numerical calculation

The selected action is **South in state 386**. Its probability under the fixed old policy is $0.40$. The learnable current policy starts with the same logits, so its initial probability is also $0.40$. The advantage is the explicitly assumed, fixed value $A=2.0$.

For this single sample, PPO uses

$$
r=\frac{\pi_\theta(a\mid s)}{\pi_{\mathrm{old}}(a\mid s)},\qquad
L_{\mathrm{CLIP}}=\min\left(rA,\;\operatorname{clip}(r,1-\epsilon,1+\epsilon)A\right),
\qquad
\ell_\pi=-L_{\mathrm{CLIP}}.
$$

Substituting our numbers gives

$$
\begin{aligned}
r&=0.40/0.40=1,\\
rA&=1\cdot 2=2,\\
\operatorname{clip}(r,0.8,1.2)&=1,\\
\operatorname{clip}(r,0.8,1.2)A&=1\cdot 2=2,\\
L_{\mathrm{CLIP}}&=\min(2,2)=2,\\
\mathrm{policy\_loss}=\ell_\pi&=-2.
\end{aligned}
$$

Gradient descent minimizes this negative objective. The ratio equals $1$ **at the initial parameters**; it changes when the current logits change. The old probability stays fixed at $0.40$.

The following cell verifies each intermediate value.

In [3]:
current_probabilities = masked_probabilities(policy_logits)
current_probability = current_probabilities[selected_action]
probability_ratio = current_probability / old_probability
clipped_ratio = torch.clamp(probability_ratio, 1.0 - epsilon, 1.0 + epsilon)
unclipped_surrogate = probability_ratio * advantage
clipped_surrogate = clipped_ratio * advantage
ppo_objective = torch.minimum(unclipped_surrogate, clipped_surrogate)
policy_loss = -ppo_objective

objective_values = [
    ("Old selected-action probability", old_probability),
    ("Current selected-action probability", current_probability),
    ("Fixed advantage", advantage),
    ("Probability ratio", probability_ratio),
    ("Unclipped surrogate", unclipped_surrogate),
    ("Clipped ratio", clipped_ratio),
    ("Clipped surrogate", clipped_surrogate),
    ("Final PPO objective", ppo_objective),
    ("Policy loss", policy_loss),
]
print("\n".join(f"{name:<36} {value.item():.6f}" for name, value in objective_values))

Old selected-action probability      0.400000
Current selected-action probability  0.400000
Fixed advantage                      2.000000
Probability ratio                    1.000000
Unclipped surrogate                  2.000000
Clipped ratio                        1.000000
Clipped surrogate                    2.000000
Final PPO objective                  2.000000
Policy loss                          -2.000000


## 4. Policy update calculated by hand

### Step 1: calculate the gradients

The initial ratio $r=1$ is inside $[0.8,1.2]$, so clipping is inactive locally and

$$
\ell_\pi=-A\frac{p_a}{p_{\mathrm{old},a}},
\qquad
\frac{\partial p_a}{\partial z_i}
=p_a\left(\mathbf{1}_{i=a}-p_i\right).
$$

The old probability and advantage are constants. For each valid action,

$$
\frac{\partial\ell_\pi}{\partial z_i}
=-Ar\left(\mathbf{1}_{i=a}-p_i\right).
$$

At $A=2$ and $r=1$, the numerical gradients are

$$
\begin{aligned}
g_{\mathrm{South}}&=2(0.40-1)=-1.20,\\
g_{\mathrm{North}}&=2(0.35-0)=0.70,\\
g_{\mathrm{West}}&=2(0.25-0)=0.50.
\end{aligned}
$$

Invalid logits are replaced by constants before softmax, so their gradients are zero. In the six-action order,

$$
g=[-1.20,\;0.70,\;0,\;0.50,\;0,\;0].
$$

### Step 2: update the logits

One SGD step with learning rate $\alpha=0.1$ gives $z'_i=z_i-0.1g_i$:

| Action | Initial logit | Calculation | Updated logit |
|---|---:|---|---:|
| South | $\log 4\approx1.386294$ | $\log 4-0.1(-1.2)=\log 4+0.12$ | $1.506294$ |
| North | $\log 3.5\approx1.252763$ | $\log 3.5-0.1(0.7)=\log 3.5-0.07$ | $1.182763$ |
| West | $\log 2.5\approx0.916291$ | $\log 2.5-0.1(0.5)=\log 2.5-0.05$ | $0.866291$ |

The invalid-action logits remain unchanged. Displayed decimals are rounded; the calculations use the full-precision logarithms.

### Step 3: recalculate the masked probabilities

The changes to the valid logits are $+0.12$, $-0.07$, and $-0.05$. We multiply the previous valid probabilities by the corresponding exponential factors and normalize:

$$
\begin{aligned}
D&=0.40e^{0.12}+0.35e^{-0.07}+0.25e^{-0.05}\\
&\approx0.450998741+0.326337837+0.237807356\\
&\approx 1.015143934.
\end{aligned}
$$

Hence,

$$
\begin{aligned}
p'_{\mathrm{South}}&=\frac{0.40e^{0.12}}{D}
\approx\frac{0.450998741}{1.015143934}\approx 0.444271,\\
p'_{\mathrm{North}}&=\frac{0.35e^{-0.07}}{D}
\approx 0.321470,\\
p'_{\mathrm{West}}&=\frac{0.25e^{-0.05}}{D}
\approx 0.234260.
\end{aligned}
$$

The updated policy is

$$
\pi_{\mathrm{after}}
\approx[0.444271,\;0.321470,\;0,\;0.234260,\;0,\;0].
$$

The exact valid probabilities sum to $1$; invalid probabilities remain exactly $0$.

### Step 4: check the objective after the update

Using the **same fixed old probability**,

$$
r_{\mathrm{after}}
=\frac{0.444270734}{0.40}\approx1.110677.
$$

This is still inside $[0.8,1.2]$, so

$$
\ell_{\pi,\mathrm{after}}
\approx-\min(1.110676835\cdot2,\;1.110676835\cdot2)
\approx-2.221354.
$$

| Quantity | Before the update | After one SGD step |
|---|---:|---:|
| South probability | $0.400000$ | $0.444271$ |
| Ratio against the fixed old policy | $1.000000$ | $1.110677$ |
| Policy loss | $-2.000000$ | $-2.221354$ |

The following PyTorch cell verifies the hand calculation using backward() and an actual optimizer step.

In [4]:
logits_before = policy_logits.detach().clone()
probabilities_before = current_probabilities.detach().clone()
optimizer = torch.optim.SGD([policy_logits], lr=learning_rate)
optimizer.zero_grad()
policy_loss.backward()
gradients = policy_logits.grad.detach().clone()
optimizer.step()

with torch.no_grad():
    logits_after = policy_logits.detach().clone()
    probabilities_after = masked_probabilities(policy_logits)
    ratio_after = probabilities_after[selected_action] / old_probability
    loss_after = -torch.minimum(
        ratio_after * advantage,
        torch.clamp(ratio_after, 1.0 - epsilon, 1.0 + epsilon) * advantage,
    )

print(
    f"{'Action':<8} {'Logit before':>13} {'Gradient':>10}"
    f" {'Logit after':>13} {'p before':>10} {'p after':>10}"
)
for i, name in enumerate(ACTION_NAMES):
    print(
        f"{name:<8} {logits_before[i].item():>13.6f} {gradients[i].item():>10.6f}"
        f" {logits_after[i].item():>13.6f}"
        f" {probabilities_before[i].item():>10.6f} {probabilities_after[i].item():>10.6f}"
    )
print(
    f"\nSouth probability: {probabilities_before[selected_action].item():.6f}"
    f" -> {probabilities_after[selected_action].item():.6f}\n"
    f"Policy loss: {policy_loss.item():.6f} -> {loss_after.item():.6f}\n"
    f"Ratio after update, against the fixed old policy: {ratio_after.item():.6f}"
)

Action    Logit before   Gradient   Logit after   p before    p after
South         1.386294  -1.200000      1.506294   0.400000   0.444271
North         1.252763   0.700000      1.182763   0.350000   0.321470
East          1.609438   0.000000      1.609438   0.000000   0.000000
West          0.916291   0.500000      0.866291   0.250000   0.234260
Pickup        1.098612   0.000000      1.098612   0.000000   0.000000
Dropoff       0.693147   0.000000      0.693147   0.000000   0.000000

South probability: 0.400000 -> 0.444271
Policy loss: -2.000000 -> -2.221354
Ratio after update, against the fixed old policy: 1.110677


### A separate calculation where clipping is active

The actual step above increases South's probability from $0.400000$ to $0.444271$. Its new ratio is about $1.110677$, still inside the clipping interval.

To show clipping clearly, consider a **separate hypothetical current policy** with valid probabilities $[0.50,\;0.30,\;0.20]$ for South, North, and West, matching notebook 03. The old South probability stays $0.40$:

$$
r=\frac{0.50}{0.40}=1.25,\quad
rA=2.50,\quad
\operatorname{clip}(r,0.8,1.2)A=1.2\cdot2=2.40,
\quad
\mathrm{policy\_loss}=-\min(2.50,2.40)=-2.40.
$$

Because $A>0$ and $r>1+\epsilon$, this sample's objective is flat and its policy gradient is zero. The following backward() verifies that case without changing the policy updated above. Clipping removes the incentive to increase this sample's probability further; it does **not** impose a hard bound on parameter changes or all action probabilities.

In [5]:
clipping_logits = torch.nn.Parameter(
    torch.log(torch.tensor([5.0, 3.0, 5.0, 2.0, 3.0, 2.0], dtype=torch.float64))
)
clipping_probabilities = masked_probabilities(clipping_logits)
clipping_probability = clipping_probabilities[selected_action]
clipping_ratio = clipping_probability / old_probability
clipping_ratio_bounded = torch.clamp(clipping_ratio, 1.0 - epsilon, 1.0 + epsilon)
clipping_unclipped_surrogate = clipping_ratio * advantage
clipping_clipped_surrogate = clipping_ratio_bounded * advantage
clipping_objective = torch.minimum(clipping_unclipped_surrogate, clipping_clipped_surrogate)
clipping_loss = -clipping_objective
clipping_loss.backward()

clipping_values = [
    ("Fixed old selected-action probability", old_probability),
    ("Hypothetical current probability", clipping_probability),
    ("Fixed advantage", advantage),
    ("Probability ratio", clipping_ratio),
    ("Unclipped surrogate", clipping_unclipped_surrogate),
    ("Clipped ratio", clipping_ratio_bounded),
    ("Clipped surrogate", clipping_clipped_surrogate),
    ("Final PPO objective", clipping_objective),
    ("Policy loss", clipping_loss),
]
print("\n".join(f"{name:<38} {value.item():.6f}" for name, value in clipping_values))
print("Gradient for the hypothetical policy:", clipping_logits.grad.tolist())

Fixed old selected-action probability  0.400000
Hypothetical current probability       0.500000
Fixed advantage                        2.000000
Probability ratio                      1.250000
Unclipped surrogate                    2.500000
Clipped ratio                          1.200000
Clipped surrogate                      2.400000
Final PPO objective                    2.400000
Policy loss                            -2.400000
Gradient for the hypothetical policy: [0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


### Numerical checks

These assertions verify normalization, exact masking, the fixed old policy and advantage, the analytical gradient, the actual SGD update, and the zero-gradient clipping case.

In [6]:
assert not old_logits.requires_grad and old_logits.grad is None
assert not old_probability.requires_grad and not advantage.requires_grad
assert torch.equal(old_logits, initial_logits)
assert torch.equal(old_probabilities, masked_probabilities(old_logits))
for probabilities in (
    old_probabilities, probabilities_before, probabilities_after, clipping_probabilities.detach()
):
    assert torch.all(probabilities >= 0)
    assert torch.all(probabilities[~action_mask] == 0)
    torch.testing.assert_close(probabilities[action_mask].sum(), torch.tensor(1.0, dtype=torch.float64))

torch.testing.assert_close(
    gradients, torch.tensor([-1.2, 0.7, 0.0, 0.5, 0.0, 0.0], dtype=torch.float64)
)
torch.testing.assert_close(logits_after, logits_before - learning_rate * gradients)
assert not torch.equal(logits_before, logits_after)
assert probabilities_after[selected_action] > probabilities_before[selected_action]
torch.testing.assert_close(
    probabilities_after[selected_action], torch.tensor(0.44427073407933526, dtype=torch.float64)
)
torch.testing.assert_close(policy_loss.detach(), torch.tensor(-2.0, dtype=torch.float64))
torch.testing.assert_close(clipping_loss.detach(), torch.tensor(-2.4, dtype=torch.float64))
assert torch.all(clipping_logits.grad == 0)
print("All numerical checks passed.")

All numerical checks passed.


## 5. Conclusion

One SGD step increases **South's probability from 0.400000 to 0.444271**, while the policy loss decreases from **-2.000000 to -2.221354**. Under the assumed positive advantage, the objective treats the selected action as better than expected and rewards increasing its probability. With a negative advantage, an unclipped update would instead decrease its probability.

PPO clipping limits the incentive for favorable probability-ratio changes: the separate $r=1.25$ example gives loss $-2.40$ and zero gradient for this sample. Action masking keeps **East, Pickup, and Dropoff at exactly zero probability** before and after the update.

This is one illustrative **policy update**. Full MaskablePPO training uses batches of collected transitions, advantage estimation such as GAE, and multiple optimization steps, together with critic and entropy terms. This example isolates the clipped policy loss and requires no value network.